# Environment Setup

In [ ]:
from pathlib import Path

train_data_file = Path("/home/ssheikhi/sahar-test-codes/CRM-Data-Automation/data/train_data.jsonl")
if not train_data_file.exists():
    raise FileNotFoundError(f"Could not find dataset file: {train_data_file}")

print(f"✅ Using dataset file: {train_data_file}")
    print("✅ train_data.jsonl is already present.")
else:
    candidate_paths = [
        Path("data/train_data.jsonl"),
        Path("../data/train_data.jsonl"),
        Path("/content/CRM-Data-Automation/data/train_data.jsonl"),
        Path("/content/data/train_data.jsonl"),
        Path("/home/ssheikhi/sahar-test-codes/CRM-Data-Automation/data/train_data.jsonl"),
        Path("/content/drive/MyDrive/CRM-Data-Automation/data/train_data.jsonl"),
    ]

    source_file = next((path for path in candidate_paths if path.exists()), None)
    if source_file is None:
        searched = "\n".join(str(path) for path in candidate_paths)
        raise FileNotFoundError(
            "Could not find train_data.jsonl automatically. Checked:\n" + searched
        )

    shutil.copy2(source_file, target_file)
    print(f"✅ Copied dataset from: {source_file.resolve()} -> {target_file.resolve()}")

Detected Colab Remote Kernel. Installing Unsloth...
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-xev6wykp/unsloth_d43a4f5a592a4eddbb2bde96023f4e05
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-xev6wykp/unsloth_d43a4f5a592a4eddbb2bde96023f4e05
  Resolved https://github.com/unslothai/unsloth.git to commit d225f45ffae256e0e467b4e9087f34074d7cb36b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.8.6
    Uninstalling trl-0.8.6:
      Successfully uninstalled trl-0.8.6
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.2

In [5]:
import os

# 1. Check for Colab GPU and install necessary libraries
if 'COLAB_GPU' in os.environ or 'KAGGLE_URL_BASE' in os.environ:
    print("Detected Remote GPU Environment. Installing Unsloth...")
    %pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    %pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
else:
    print("Warning: No remote GPU detected. Ensure you are connected to the Colab T4 Kernel.")

Detected Remote GPU Environment. Installing Unsloth...
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-_bxbjx7k/unsloth_f5eaf2766a98406b8f4caf26dde573b3
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-_bxbjx7k/unsloth_f5eaf2766a98406b8f4caf26dde573b3
  Resolved https://github.com/unslothai/unsloth.git to commit d225f45ffae256e0e467b4e9087f34074d7cb36b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.8.6
    Uninstalling trl-0.8.6:
      Successfully uninstalled trl-0.8.6
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  Attempting uninstall: trl
    Found existing installation: trl 

# Data Upload

In [6]:
from google.colab import files

# 2. Upload your generated data
if not os.path.exists('train_data.jsonl'):
    print("Please upload your data/train_data.jsonl file:")
    uploaded = files.upload()
else:
    print("✅ train_data.jsonl is already present on the server.")

Please upload your data/train_data.jsonl file:


KeyboardInterrupt: 

# Load Model & Tokenizer

In [ ]:
from unsloth import FastLanguageModel
import torch

# 3. Configuration and Loading
max_seq_length = 2048 
dtype = None # Auto detection
load_in_4bit = True # Reduces memory usage by 4x

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA Adapters

In [ ]:
# 4. Set up Parameter-Efficient Fine-Tuning (PEFT)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: higher = more parameters but more memory
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized to 0 for speed
    bias = "none",    # Optimized to "none" for speed
    use_gradient_checkpointing = "unsloth", # Saves massive VRAM
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# Data Formatting

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# 5. Format the dataset for Llama-3.2
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.2",
)

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        parts = [
            {"role": "user", "content": f"{instruction}\n\nInput: {input}"},
            {"role": "assistant", "content": output}
        ]
        text = tokenizer.apply_chat_template(parts, tokenize = False, add_generation_prompt = False)
        texts.append(text)
    return { "text" : texts, }

dataset = load_dataset("json", data_files={"train": "train_data.jsonl"}, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

print("Example formatted prompt:")
print(dataset[0]["text"])

# The Training Configuration (SFTTrainer)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# 6. Initialize the Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Set to 60 for a fast 'proof of concept'
        # num_train_epochs = 1, # Use this for one full pass of your data
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)